In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap

import fig_settings as fs

fs.set_colors()
fs.set_fonts()

In [ ]:
RESULTS = "../results"
counts = np.load("../data/wechat/processed/counts.npz")
fit1 = np.load(f"{RESULTS}/fits/wechat/K~1.npz")
fit2 = np.load(f"{RESULTS}/fits/wechat/K~2.npz")
cx = pd.read_parquet(f"{RESULTS}/complexity/wechat/complexity.parquet").set_index(["K", "component"])["complexity"]
sbm = np.load(f"{RESULTS}/communities/wechat/sbm.npz")
edges = np.loadtxt("../data/wechat/processed/wechat.edgelist", dtype=int)

M, N = counts["M"], counts["N"]
p_low = fit2["post_prob"][:, 0]
kmax = M.shape[1]
nus = np.arange(1, kmax + 1)

In [ ]:
def _hpdi(x, prob=0.5):
    """Narrowest interval containing `prob` of 1D samples."""
    x = np.sort(x)
    n = len(x)
    k = int(np.floor(prob * n))
    i = np.argmin(x[k:] - x[:n - k])
    return x[i], x[i + k]

def hpdi_bounds(X, prob=0.5):
    lower, upper = np.empty(X.shape[1]), np.empty(X.shape[1])
    for j in range(X.shape[1]):
        lower[j], upper[j] = _hpdi(X[:, j], prob=prob)
    return lower, upper

def wilson(m, t, z=1.96):
    p = m / t
    c = (p + z**2 / (2 * t)) / (1 + z**2 / t)
    h = z * np.sqrt(p * (1 - p) / t + z**2 / (4 * t**2)) / (1 + z**2 / t)
    return c - h, c + h

def plot_mu_with_shading(ax, mu_array, color, zorder):
    ax.plot(nus, mu_array.mean(axis=0), color=color, lw=2.0, zorder=zorder)
    lower, upper = hpdi_bounds(mu_array, prob=0.9)
    ax.fill_between(nus, lower, upper, color=color, alpha=0.2, lw=0, zorder=zorder)

def plot_rates(ax, m, t, color, x=nus):
    lower, upper = wilson(m, t)
    ax.vlines(x, lower, upper, color=color, lw=0.6, alpha=0.35, zorder=1)
    ax.plot(x, m / t, "o", color=color, ms=2.5, zorder=4, clip_on=False)

In [ ]:
# friend network: largest connected component, blocks placed on a circle, spring layout inside
nodes, block = sbm["nodes"], sbm["block"]
G = nx.Graph([tuple(e) for e in edges]).subgraph(nodes.tolist())
B = block.max() + 1
centre = np.c_[np.cos(2 * np.pi * np.arange(B) / B), np.sin(2 * np.pi * np.arange(B) / B)]
rng = np.random.default_rng(0)
init = {int(n): centre[b] + 0.25 * rng.standard_normal(2) for n, b in zip(nodes, block)}
pos = nx.spring_layout(G, pos=init, k=0.06, iterations=60, seed=0)
xy = np.array([pos[int(n)] for n in nodes])
idx = {int(n): i for i, n in enumerate(nodes)}
E = np.array([[idx[u], idx[v]] for u, v in G.edges()])

In [ ]:
c_high, c_low, c_one = fs.palette[2], fs.palette[5], fs.palette[3]
cmap = LinearSegmentedColormap.from_list("p_low", [c_high, c_low])
min_eff = 10  # responsibility-weighted levels with fewer effective trials are not drawn

fig = plt.figure(figsize=(3.375, 3.9))
gs = fig.add_gridspec(2, 2, height_ratios=[1.6, 0.8], left=0.15, right=0.97, top=0.93, bottom=0.1, wspace=0.14, hspace=0.45)
axs = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, :])]

# PANEL A - K = 1 against the pooled data
plot_rates(axs[0], M.sum(0), (M + N).sum(0), "dimgrey")
plot_mu_with_shading(axs[0], fit1["draws_mu"][:, 0], c_one, 2)
axs[0].text(1.5, 0.03, f"$D={cx[1, 1]:.1f}$", color=c_one, fontsize=8)

# PANEL B - K = 2, data split by each user's responsibility, network inset
plot_mu_with_shading(axs[1], fit2["draws_mu"][:, 1], c_high, 2)
plot_mu_with_shading(axs[1], fit2["draws_mu"][:, 0], c_low, 3)
t = M + N
for r, color in [(p_low, c_low), (1 - p_low, c_high)]:
    m_k, t_k = (r[:, None] * M).sum(0), (r[:, None] * t).sum(0)
    keep = t_k >= min_eff
    plot_rates(axs[1], m_k[keep], t_k[keep], color, x=nus[keep])
axs[1].text(1.5, 0.45, f"$D={cx[2, 2]:.1f}$", color=c_high, fontsize=8)
axs[1].text(6.5, 0.03, f"$D={cx[2, 1]:.1f}$", color=c_low, fontsize=8)

ins = axs[1].inset_axes([0.5, 0.0, 0.5, 0.44])
ins.add_collection(LineCollection(xy[E], colors="silver", lw=0.2, zorder=1))
o = np.argsort(np.abs(p_low[nodes] - 0.5))
ins.scatter(xy[o, 0], xy[o, 1], c=p_low[nodes][o], cmap=cmap, vmin=0, vmax=1, s=1.2, lw=0, zorder=2)
ins.set_aspect("equal")
ins.axis("off")

axs[1].sharey(axs[0])
axs[1].tick_params(labelleft=False)
for ax in axs[:2]:
    ax.set_xlim(1, kmax)
    ax.set_ylim(0, 1)
    ax.set_xticks([1, 10, 20])
    ax.set_yticks([0.0, 0.5, 1.0])
    ax.set_xlabel("Exposures, $\\nu$")
axs[0].set_ylabel("Share prob., $c(\\nu)$")

# PANEL C - mean P(low class) per SBM block against the permutation null
y = np.arange(len(sbm["size"]))[::-1]
axs[2].hlines(y, *sbm["null_band"].T, color="silver", lw=2.5, alpha=0.35)
axs[2].scatter(sbm["obs"], y, c=sbm["obs"], cmap=cmap, vmin=0, vmax=1, s=22, lw=0, zorder=3)
axs[2].axvline(float(sbm["overall"]), color="grey", ls="--", lw=1, zorder=0)
tk = np.unique([0, (len(y) - 1) // 2, len(y) - 1])
axs[2].set_yticks(y[tk], tk + 1)
axs[2].set_ylim(-0.7, len(y) - 0.3)
axs[2].set_xlim(0, 1)
axs[2].set_xticks([0.0, 0.5, 1.0])
axs[2].set_xlabel("Mean P(low class)")
axs[2].set_ylabel("SBM block")
axs[2].legend(
    handles=[plt.Line2D([], [], color="silver", lw=4, alpha=0.35), plt.Line2D([], [], ls="", marker="o", ms=4.5, color="dimgrey")],
    labels=["Global shuffle 90%", "Observed"],
    loc="lower right", bbox_to_anchor=(1.0, 1.0), ncol=2, frameon=False, handlelength=1.2, borderaxespad=0.1, columnspacing=1.0,
)

for ax, s in zip(axs, "abc"):
    ax.text(-0.15 if s != "c" else -0.07, 1.1, f"({s})", transform=ax.transAxes, fontsize=8, fontweight="bold", va="center", ha="left")

plt.savefig("../figs/fig5.svg")
plt.savefig("../figs/fig5.pdf")
plt.show()

In [ ]:
# numbers quoted in the text
from glob import glob

folds = pd.concat(pd.read_parquet(f) for f in glob(f"{RESULTS}/cv/wechat/*_fold.parquet"))
lppd = folds.pivot(index="node_idx", columns="K", values="log_pred")
diff = lppd[2] - lppd[1]
print(f"held-out lppd K=2 - K=1: {diff.mean():.3f} +- {diff.std() / np.sqrt(len(diff)):.3f} nats per user (paired SE)")
print(cx)
outside = (sbm["obs"] < sbm["null_band"][:, 0]) | (sbm["obs"] > sbm["null_band"][:, 1])
print(f"{len(sbm['nodes'])} users in the LCC, {len(sbm['size'])} SBM blocks, {outside.sum()} outside the null band")